# Clean Global Force Model — TEST2 to TEST6

## tl;dr

- The model uses **45 complete sessions**, five repeats at every ROI, and **15,894 frames** collected with saturation 54 and sharpness 38.
- It is global: no ROI label is used.
- Entire numbered test runs are held out during validation.
- Best result: **MAE 1.65 N**, **RMSE 2.13 N**, **R² 0.362** with a one-frame output delay.
- It is suitable as an experimental force estimator after contact is detected, but it cannot reliably decide zero/no-contact because matching no-contact recordings were not available for this camera configuration.


## Context & Methods

The source is the updated `Manual Calibration.zip`. TEST1, TEST7, and NOCONTACT are deliberately excluded because their saturation or sharpness settings differ from TEST2–TEST6.

### Key Assumptions

- Negative load-cell readings are zeroed to 0 N.
- The force estimate may lag by one camera frame because this alignment validated better than zero-lag prediction.
- The model is evaluated by leaving out all nine ROI sessions belonging to one complete test number, then repeating for TEST2 through TEST6.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'clean_global_force_model').exists():
    ROOT = Path(r'C:\Users\DLSU\OneDrive\Documents\SOFTWARE CALIBRATION\analysis_outputs')
MODEL_DIR = ROOT / 'clean_global_force_model'
model = json.loads((MODEL_DIR / 'clean_global_force_model.json').read_text())
metrics = json.loads((MODEL_DIR / 'validation_metrics.json').read_text())
comparison = pd.read_csv(MODEL_DIR / 'model_comparison.csv', index_col=0)
per_roi = pd.read_csv(MODEL_DIR / 'per_roi_metrics.csv')
oof = pd.read_csv(MODEL_DIR / 'out_of_session_predictions.csv.gz')
model['training_scope']


{'archive': 'C:\\Users\\DLSU\\Downloads\\Data Collection\\Manual Calibration.zip', 'included_runs': [2, 3, 4, 5, 6], 'sessions': 45, 'frames': 15894, 'sessions_per_roi': {'1': 5, '2': 5, '3': 5, '4': 5, '5': 5, '6': 5, '7': 5, '8': 5, '9': 5}, 'force_range_N': [0.0, 19.856225637058383]}


## Data

### 1. Included scope

In [2]:
pd.Series({
    'sessions': model['training_scope']['sessions'],
    'frames': model['training_scope']['frames'],
    'minimum force (N)': model['training_scope']['force_range_N'][0],
    'maximum force (N)': model['training_scope']['force_range_N'][1],
    'included test numbers': model['training_scope']['included_runs'],
    'sessions per ROI': sorted(set(model['training_scope']['sessions_per_roi'].values())),
})


sessions                              45
frames                             15894
minimum force (N)                    0.0
maximum force (N)              19.856226
included test numbers    [2, 3, 4, 5, 6]
sessions per ROI                     [5]
dtype: object


## Results

### 2. Candidate comparison

In [3]:
comparison[['mae_N', 'rmse_N', 'r2', 'bias_N', 'target_delay_frames']]


                                          mae_N  ...  target_delay_frames
total_light_linear                     1.984324  ...                  0.0
global_multichannel_linear             1.707601  ...                  0.0
global_nonlinear_basis                 1.675998  ...                  0.0
global_nonlinear_basis_1frame_delayed  1.646169  ...                  1.0

[4 rows x 5 columns]


### 3. Complete-run holdouts

In [4]:
pd.DataFrame(metrics['selected_model_fold_results']).set_index('held_out_test')


               frames     mae_N    rmse_N        r2    bias_N
held_out_test                                                
2                3396  1.817595  2.332967  0.345447 -0.425637
3                3129  1.807603  2.436853  0.325733 -0.190880
4                3236  1.508866  1.897679  0.382817  0.385981
5                3016  1.580303  1.992986  0.336714  0.231990
6                3072  1.501532  1.906403  0.403881  0.001994


### 4. Performance by physical ROI

In [5]:
per_roi.set_index('roi')[['mae_N', 'rmse_N', 'r2', 'bias_N']]


        mae_N    rmse_N        r2    bias_N
roi                                        
1    2.344076  3.097603  0.178217 -0.886255
2    1.940903  2.506347  0.132947 -0.341105
3    1.647910  2.172584  0.262684  0.101255
4    1.738772  2.093596  0.265892  0.417378
5    1.653297  1.965319  0.223062  0.298595
6    1.452743  1.849664  0.173555  0.174844
7    1.250321  1.555281  0.204374  0.251861
8    1.394503  1.758451  0.213351 -0.006959
9    1.308036  1.607879  0.124189 -0.094369


The model is materially better than the earlier one-session-per-ROI analysis. Performance is reasonably consistent across held-out test numbers, but ROI 1 remains the weakest location with MAE 2.34 N. This is useful engineering evidence that the added repeats helped.

The low-force check still fails: all held-out frames below 0.05 N receive predictions above the contact threshold. This occurs because the compatible TEST2–TEST6 data contains relatively little sustained zero-force data. Use an independent contact detector and only display the force estimate after contact.


## Takeaways

In [6]:
checks = {
    '45 sessions included': model['training_scope']['sessions'] == 45,
    'five sessions per ROI': set(model['training_scope']['sessions_per_roi'].values()) == {5},
    'all intended tests represented': set(oof.test_number.unique()) == {2, 3, 4, 5, 6},
    'one prediction per usable row': oof.loc[oof.usable, 'prediction_N'].notna().all(),
    'coefficients match features': len(model['model']['feature_names']) == len(model['model']['standardized_coefficients']),
    'finite predictions': np.isfinite(oof.loc[oof.usable, 'prediction_N']).all(),
}
assert all(checks.values())
checks


{'45 sessions included': True, 'five sessions per ROI': True, 'all intended tests represented': True, 'one prediction per usable row': np.True_, 'coefficients match features': True, 'finite predictions': np.True_}


1. Integrate this as a **contact-phase global force estimator**, not as the contact detector.
2. Enforce the recorded camera requirements—especially saturation 54 and sharpness 38.
3. Collect matching no-contact data before expecting the model to output a reliable zero.
4. Keep the one-frame delay explicit in the user interface and do not use this estimate for emergency force limits.
